In [1]:
%load_ext autoreload
%autoreload 2

from math import pi
import os
from typing import NamedTuple


os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


import matplotlib.pyplot as plt
import numpy as np
import tqdm.contrib
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.dressed_control_fluxonium import (
    create_driven_fluxonium,
    calculate_critical_amplitude,
    calculate_polarization_and_error,
    calculate_optimal_amplitude,
)
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.utils import load_arguments

In [40]:
parent_path = DATA_DIR/"polarization_spectrum/EJ=4.0,EC=1.0,EL=1.0"

argm = load_arguments(parent_path)

fx = Fluxonium(
    EJ=argm.EJ,
    EC=argm.EC,
    EL=argm.EL,
    dim=argm.dim,
    cutoff=argm.cutoff
)
qubit_frequency = fx.eigenvalues[1] - fx.eigenvalues[0]
n_op = fx.get_operator('charge')
Ω0 = qubit_frequency/abs(n_op[0, 1])

lookup_amps = Ω0 * argm.lookup_amplitudes
lookup_deg_tol = argm.lookup_degeneracy_tol

In [41]:
dataset = xr.Dataset()

frequency_coord = xr.DataArray(
    argm.frequencies,
    dims=['frequency'],
    attrs=dict(
        units="Grad/s",
    ),
)

dataset['p0'] = xr.DataArray(
    np.nan,
    coords=[frequency_coord],
)
dataset['p1'] = xr.DataArray(
    np.nan,
    coords=[frequency_coord],
)
dataset['error'] = xr.DataArray(
    np.nan,
    coords=[frequency_coord],
)
dataset['amplitude'] = xr.DataArray(
    np.nan,
    coords=[frequency_coord],
    attrs=dict(
        units="Grad/s",
        plot_unit=Ω0,
    )
)

In [43]:
from multiprocessing.pool import Pool


def doit(drive_freq):
    floquet_basis = create_driven_fluxonium(
        fx,
        drive_freq,
    )
    floquet_basis.generate_lookup(
        lookup_amps,
        deg_tol=lookup_deg_tol,
    )
    
    if drive_freq > qubit_frequency:
        x1 = Ω0
    else:
        x1 = None
    
    amp, p0, p1, err = calculate_optimal_amplitude(
        fx,
        floquet_basis,
        step_size=0.01 * Ω0,
        x1=x1,
    )
    return amp, p0, p1, err


with Pool(processes=8) as pool:
    for drive_freq, (amp, p0, p1, err) in tqdm.contrib.tzip(
            dataset.frequency.data,
            pool.imap(doit, dataset.frequency.data),
    ):
        ds = dataset.loc[dict(frequency=drive_freq)]
        ds['amplitude'][()] = amp
        ds['p0'][()] = p0
        ds['p1'][()] = p1
        ds['error'][()] = err

  0%|          | 0/5801 [00:00<?, ?it/s]

In [55]:
dataset.to_netcdf(parent_path/"dataset.hdf5",)